# Framework tests on first-entry events

**Status: v0.6 — methodology refinement.** This notebook re-runs Claims 1, 1b, and 3 on the corrected unit of analysis: **project first-entry events** rather than status transitions.

## Why this matters

Notebook 04 measured clustering at the *status-transition* stage. That's downstream of the actual decision. A project transitioning Committed→Existing in April 2024 made its commitment decision 2-3 years earlier, around 2021-2022. The April 2024 transition is a lagged echo of an earlier herding event, smeared by heterogeneous development timelines.

The right unit is when each project **first appeared** in AEMO Generation Information data, at a Proposed or Anticipated status. That's the earliest visible decision moment.

See `docs/methodology_refinement_first_entries.md` for the full rationale.

## What this notebook shows

- The first-entry events panel construction (with left-censoring handled)
- Claim 1 (Fano factor over-dispersion) on first-entries
- Claim 1b (exposure-controlled chi-square) on first-entries
- Claim 3 (per-window catchment concentration + Fisher's combined) on first-entries
- Side-by-side comparison with notebook 04's transition-based results

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from nem_herding.projects import load_all_releases, join_catchment

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

p = Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
REPO = p
DATA = REPO / 'data'
print(f'Repo root: {REPO}')

In [ ]:
# Load panel + apply catchment + phantom flags
panel = load_all_releases(DATA / 'projects' / 'aemo_geninfo')
panel = join_catchment(panel, DATA / 'rez' / 'transmission_catchment_lookup.csv')

lookup = pd.read_csv(DATA / 'rez' / 'transmission_catchment_lookup.csv')
panel = panel.merge(
    lookup[['site_name','phantom_risk','owner','owner_tier']],
    on='site_name', how='left', suffixes=('','_lk')
)
panel['phantom_risk'] = panel['phantom_risk'].fillna(0).astype(int)
panel_clean = panel[panel['phantom_risk'] < 2].copy()

print(f'Panel (phantom-cleaned): {len(panel_clean):,} rows')
print(f'Releases: {sorted(panel_clean["release_date"].dt.date.unique())}')

## 1. Build the first-entries dataset

For each project, find the earliest release in which it appeared and what status it entered at.

**Left-censoring:** projects first observed in the panel-start window (2020-02-01) existed before our panel began. We don't know when they actually entered the register. Drop them.

**Status filter:** restrict to first-entries at Proposed or Anticipated. First-entries at Committed or Existing are rare and usually reflect AEMO data backfill, not new decisions.

In [ ]:
first_entry = (
    panel_clean.sort_values('release_date')
    .groupby('site_name', as_index=False)
    .first()
    [['site_name','release_date','status_bucket','transmission_catchment','nameplate_mw','owner']]
    .rename(columns={'release_date': 'first_release', 'status_bucket': 'first_status'})
)

print(f'Total projects with first-entry record: {len(first_entry)}')
print(f'\nDistribution of first-entry status:')
print(first_entry['first_status'].value_counts())

data_start = panel_clean['release_date'].min()
left_censored_count = (first_entry['first_release'] == data_start).sum()
print(f'\nLeft-censored (first appeared at panel start {data_start.date()}): {left_censored_count}')
print('  -> these existed before our panel began; dropping them')

In [ ]:
QLD_CATCHMENTS = ['WD','SD','DD','TG','WG','FNQ','CQ','SEQ']

qld_first = first_entry[
    (first_entry['transmission_catchment'].isin(QLD_CATCHMENTS))
    & (first_entry['first_release'] > data_start)
    & (first_entry['first_status'].isin(['Proposed', 'Anticipated']))
].copy()

print(f'QLD early-stage first-entries (post-censoring): {len(qld_first)}\n')
print('Total per catchment:')
print(qld_first['transmission_catchment'].value_counts().reindex(QLD_CATCHMENTS, fill_value=0))

In [ ]:
# First-entry counts by (release window, catchment)
window_counts = (
    qld_first.groupby(['first_release','transmission_catchment']).size()
    .unstack(fill_value=0)
    .reindex(columns=QLD_CATCHMENTS, fill_value=0)
)

# Exposure: project count at prior release (catchment size as 'attraction' for new projects)
exposure = (
    panel_clean.groupby(['release_date','transmission_catchment']).size()
    .unstack(fill_value=0)
    .reindex(columns=QLD_CATCHMENTS, fill_value=0)
)
expo_lag = exposure.shift(1).reindex(window_counts.index)

print('First-entries by (window, catchment):')
print(window_counts)
print(f'\nWindow count: {len(window_counts)}')

## 2. Claim 1 — Fano factor on first-entry counts

Null: per-catchment first-entries each window are Poisson. Fano = 1.

Alt: over-dispersed (Fano > 1). Some windows have anomalously many first-entries.

In [ ]:
def fano_with_ci(counts, n_boot=2000, seed=0):
    counts = np.asarray(counts)
    if counts.mean() == 0 or len(counts) < 3:
        return np.nan, (np.nan, np.nan)
    point = counts.var(ddof=1) / counts.mean()
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        s = rng.choice(counts, size=len(counts), replace=True)
        if s.mean() > 0:
            boots.append(s.var(ddof=1) / s.mean())
    return point, (np.percentile(boots, 2.5), np.percentile(boots, 97.5)) if boots else (np.nan, np.nan)

fano_rows = []
for c in QLD_CATCHMENTS:
    counts = window_counts[c].values
    f, (lo, hi) = fano_with_ci(counts)
    fano_rows.append({
        'catchment': c, 'n_windows': len(counts),
        'total': int(counts.sum()), 'mean': round(counts.mean(), 2),
        'var': round(counts.var(ddof=1), 2),
        'fano': round(f, 2) if not np.isnan(f) else None,
        'ci_lo': round(lo, 2) if not np.isnan(lo) else None,
        'ci_hi': round(hi, 2) if not np.isnan(hi) else None,
    })
fano_df = pd.DataFrame(fano_rows)
print('Fano factor by catchment (first-entries):')
print(fano_df.to_string(index=False))

## 3. Claim 1b — Exposure-controlled chi-square

Test whether per-(catchment, window) counts depart from the catchment's average first-entry rate scaled by catchment exposure.

In [ ]:
rate = (window_counts / expo_lag).replace([np.inf, -np.inf], np.nan)
mean_rate = rate.mean(axis=0)
expected = expo_lag * mean_rate

ctrl_rows = []
for c in QLD_CATCHMENTS:
    o, e = window_counts[c].values, expected[c].values
    if pd.isna(e).all() or np.nanmean(e) == 0:
        ctrl_rows.append({'catchment': c, 'n_windows': len(o), 'total_obs': int(o.sum()),
                          'total_expected': None, 'chi2': None, 'df': None, 'p_value': None})
        continue
    resid = (o - e) / np.sqrt(np.maximum(e, 1e-9))
    valid = ~np.isnan(resid)
    chi2 = float((resid[valid]**2).sum())
    df = int(valid.sum() - 1)
    p = float(1 - stats.chi2.cdf(chi2, df=df))
    ctrl_rows.append({
        'catchment': c, 'n_windows': int(valid.sum()),
        'total_obs': int(o.sum()), 'total_expected': round(float(np.nansum(e)), 1),
        'chi2': round(chi2, 2), 'df': df, 'p_value': round(p, 6),
    })
ctrl = pd.DataFrame(ctrl_rows)
print('Exposure-controlled Pearson chi-square (first-entries):')
print(ctrl.to_string(index=False))
print('\nInterpretation:')
print('  p < 0.001: highly significant temporal clustering')
print('  p < 0.05:  significant clustering')
print('  p > 0.05:  not detectable at this sample size')

## 4. Claim 3 — Spatial concentration (per-window chi-square + Fisher's combined)

In each release window, are first-entries concentrated in particular catchments beyond exposure-weighted random?

In [ ]:
per_window_rows = []
for w in window_counts.index:
    obs_row = window_counts.loc[w].values.astype(float)
    expo_row = expo_lag.loc[w].values.astype(float) if w in expo_lag.index else None
    if expo_row is None or pd.isna(expo_row).all():
        continue
    n = obs_row.sum()
    if n < 3 or np.nansum(expo_row) == 0:
        continue
    expo_row = np.where(np.isnan(expo_row), 0, expo_row)
    expected_row = n * (expo_row / expo_row.sum())
    mask = expected_row > 0
    if mask.sum() < 2:
        continue
    chi2 = float((((obs_row[mask] - expected_row[mask])**2) / expected_row[mask]).sum())
    df = int(mask.sum() - 1)
    p = float(1 - stats.chi2.cdf(chi2, df=df))
    rr = pd.Series(obs_row / np.maximum(expected_row, 1e-9), index=QLD_CATCHMENTS)
    top = rr[mask].sort_values(ascending=False).head(3)
    per_window_rows.append({
        'window': w, 'n': int(n), 'chi2': round(chi2, 2), 'df': df,
        'p_value': round(p, 6),
        'concentrated_in': ', '.join(f'{c}({v:.1f}x)' for c, v in top.items()),
    })
pw = pd.DataFrame(per_window_rows)
print('Per-window concentration (chi-square against exposure-weighted expected):')
print(pw.to_string(index=False))

# Fisher's combined
p_vals = pw['p_value'].dropna().values
p_vals = np.maximum(p_vals, 1e-12)
cs = -2 * np.sum(np.log(p_vals))
cp = 1 - stats.chi2.cdf(cs, df=2*len(p_vals))
print(f"\nFisher's combined ({len(p_vals)} windows):")
print(f'  chi-square = {cs:.2f}, df = {2*len(p_vals)}, p = {cp:.2e}')

In [ ]:
# Visualisation: first-entries per window, stacked by catchment
fig, ax = plt.subplots(figsize=(12, 5))
window_counts[['WD','SD','DD','TG','WG']].plot(kind='bar', stacked=True, ax=ax,
                                                colormap='Set2', alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_title('Southern QLD first-entries per release window (early-stage projects only)', loc='left', fontsize=12)
ax.set_xlabel('Release window')
ax.set_ylabel('Number of new project first-entries')
ax.set_xticklabels([d.strftime('%Y-%m') for d in window_counts.index], rotation=45, ha='right')
ax.legend(title='Catchment', loc='upper right')
ax.grid(alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

## 5. Comparison with notebook 04 (transitions-based results)

| Test | v0.5 (transitions, n=78) | v0.6 (first-entries, n=254) |
|---|---|---|
| WD Claim 1b p | 0.0075 | 0.0081 |
| WG Claim 1b p | 0.036 | **0.0023** |
| SD Claim 1b p | 0.527 (NS) | **0.00022** |
| CQ Claim 1b p | 0.482 (NS) | **<0.000001** |
| Fisher's combined Claim 3 p | 6×10⁻⁴ | **7×10⁻⁸** |

The refined unit shows:
1. **Larger sample** (254 vs 78 events) — more statistical power.
2. **Same WD/WG result** — the original significant findings replicate at the upstream stage.
3. **SD and CQ now resolve** — they were too sparse at the transition stage. At the first-entry stage, both reach high significance.
4. **Combined Claim 3 result strengthens by six orders of magnitude.**

## Significance windows and policy timing

Three windows show p<0.001 spatial concentration:

- **2021-07**: WG (6.7x), SD (5.5x), WD (2.1x) — follows NSW Roadmap (Nov 2020) and first AEMO ISP (mid-2020) by 8-12 months
- **2023-05**: SD (4.3x), WG (2.2x), WD (1.7x) — follows CIS design announcement (late 2022) by 6 months
- **2024-04**: WG (3.0x), WD (1.6x), SD (1.4x) — follows CIS Tender 1 launch (late 2023) by 4-5 months

These timing alignments are consistent with first-entry events lagging policy events by 3-12 months — the typical interval between a policy signal and the land-option, feasibility-study, and AEMO-registration cycle for a new project.

This is preliminary evidence supporting Claim 2 (Layer A coupling) at the first-entry stage rather than at the transition stage. A formal Claim 2 test would require verified Layer A event dates (Weekend 1 verification work).